# Concurrent Orchestration ဖြင့် ခရီးသွားအကြံပြုချက်များ

ဤစာရေးအိတ်တွင် Microsoft Agent Framework ကို အသုံးပြု၍ **concurrent orchestration** ကို ပြသပါသည်။ ခရီးသွားအချက်အလက်များကို လုံးဝမှတ်သားစေသော ခရီးသွားအကြံပြုချက်စနစ်တစ်ခုကို သုံးသီးသုံးရာ စွမ်းရည်ပြည့်ဝသော agent သုံးဦးက parallel အဖြစ် အလုပ်လုပ်ပေးမည်ဖြစ်သည်။

## သင်မည်သည်များကို သင်ယူပါမည်
1. **Concurrent Orchestration**: agent များများစွာကို parallel (fan-out/fan-in pattern) ဖြင့် တပြိုင်နက်တွင် အလုပ်လုပ်ခြင်း
2. **ConcurrentBuilder**: concurrent workflow များ ဖန်တီးရာအတွက် အဆင့်မြင့် API
3. **ခရီးသွားအကြံပြုချက်များ**: အထူးပြု agent သုံးဦး အတူတကွ အလုပ်လုပ်ခြင်း
4. **Default Aggregation**: agent များ၏ အဖြေများကို ပေါင်းစပ်ခြင်း
5. **စွမ်းဆောင်ရည် အကျိုးကျေးဇူးများ**: parallel အပြုအလုပ်မှု နှင့် တစ်ခုချင်းစီ လုပ်ဆောင်ခြင်း၏ ခြားနားချက်များ


## အထူးပြု Agent သုံးဦး

1. **Attractions Agent**: ခရီးသွားစိတ်ဝင်စားစရာနေရာများ၊ လှုပ်ရှားမှုများ၊ သမိုင်းဒေသဆိုင်ရာနေရာများ
2. **Dining Agent**: ဒေသဆိုင်ရာ ဟင်းချက်ရည်များ၊ စားသောက်ဆိုင်များ၊ အစားအသောက် အတွေ့အကြုံများ
3. **History Agent**: သမိုင်းအချက်အလက်များ၊ ယဉ်ကျေးမှု အရေးပါမှုများ၊ အကြောင်းအရာများ


In [ ]:
import asyncio
import json
import os
from typing import Any, cast

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("All imports successful!")

## Step 1: ဖွဲ့စည်းထားသော output များအတွက် Pydantic မော်ဒယ်များသတ်မှတ်ပါ

ဤမော်ဒယ်များသည် အထူးပြုထားသော agent တစ်ခုချင်းစီမှ ပြန်လည်ပေးပို့မည့် schema ကို သတ်မှတ်သည်။ ဒါကြောင့် အားလုံးသော agent များမှ တစ်ဦးချင်း ထုတ်ပေးမည့်ဖြေကြောင်းများသည် တစ်မျိုးတည်းဖြစ်ပြီး ပေါင်းသိုလှိုင်းနိုင်စေရန် အာမခံသည်။


## Step 1: ဖွဲ့စည်းရေးထားသော အထွက်များအတွက် Pydantic မော်ဒယ်များကို သတ်မှတ်ပါ

ဤမော်ဒယ်များသည် အထူးပြု agent တစ်ခုချင်းစီမှ ပြန်လည်ပေးအပ်မည့် schema ကို သတ်မှတ်ပေးသည်။ ဤသည်က အကျုံးဝင်သောနှင့် အနက်ထွင်းဆန်းစစ်နိုင်သော တုံ့ပြန်မှုများကို agent အားလုံးမှ သေချာစေရန်ဖြစ်သည်။


In [ ]:
class AttractionsRecommendation(BaseModel):
    """Tourist attractions and activities recommendations."""

    destination: str
    top_attractions: list[str]
    activities: list[str]
    best_time_to_visit: str
    transportation_tips: str  


class DiningRecommendation(BaseModel):
    """Food and dining recommendations."""

    destination: str
    local_cuisine: str
    must_try_dishes: list[str]
    recommended_restaurants: list[str]
    food_experiences: list[str]
    dining_etiquette: str


class HistoryRecommendation(BaseModel):
    """Historical and cultural information."""

    destination: str
    historical_significance: str
    cultural_highlights: list[str]
    important_periods: list[str]
    cultural_experiences: list[str]
    interesting_facts: list[str]

## Step 2: ပတ်ဝန်းကျင်ပြောင်းလဲမှုများကိုတင်ပြီး Foundry Provider ကိုဖြည့်ဆည်းပါ

သင်ခန်းစာ 01–13 တွင်အသုံးပြုသောပုံစံနှင့်အညီ keyless `AzureCliCredential` အတည်ပြုမှုဖြင့် `AzureAIProjectAgentProvider` ကိုအသုံးပြုပါ။


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("Azure AI Foundry provider configured successfully!")

## အဆင့် ၃: သီးသန့် ခရီးသွား လုပ်ငန်း လုပ်ကိုင်သူ သုံးဦး ဖန်တီးပါ။


In [ ]:
# Agent 1: Tourist Attractions Expert
attractions_agent = await provider.create_agent(
    name="attractions-agent",
    instructions=(
        "You are a tourism expert specializing in attractions and activities. "
        "When given a travel destination, provide comprehensive recommendations for "
        "tourist attractions, activities, best times to visit, and transportation tips. "
        "Focus on popular landmarks, unique experiences, and practical travel advice. "
        "Return structured JSON matching the AttractionsRecommendation schema."
    ),
)

# Agent 2: Food and Dining Expert
dining_agent = await provider.create_agent(
    name="dining-agent",
    instructions=(
        "You are a culinary expert specializing in local food and dining experiences. "
        "When given a travel destination, provide recommendations for local cuisine, "
        "must-try dishes, recommended restaurants, and unique food experiences. "
        "Include dining etiquette and cultural food customs. "
        "Return structured JSON matching the DiningRecommendation schema."
    ),
)


# Agent 3: History and Culture Expert
history_agent = await provider.create_agent(
    name="history-agent",
    instructions=(
        "You are a historian and cultural expert. "
        "When given a travel destination, provide historical context, cultural significance, "
        "important historical periods, cultural experiences, and interesting facts. "
        "Focus on helping travelers understand the cultural heritage and historical importance. "
        "Return structured JSON matching the HistoryRecommendation schema."
    ),
)

# အဆင့် ၄: အချိန်ပြိုင် Workflow တည်ဆောက်ခြင်း

`WorkflowBuilder` ကို သေးငယ်သော dispatcher executor နှင့် `add_fan_out_edges` နှင့်အတူ:
1. **Dispatcher** သည် တူညီသော εισပ်ုတ်ကို သုံးယ့်ကိုယ်ပိုင်အေဂျင့် သုံးဦးအားလုံးသို့ ထောက်ပံ့ပေးသည်
2. **အေဂျင့် သုံးဦး** ကို အချိန်ပြိုင် တပြိုင်နက် အလုပ်လုပ်သည်
3. **ရလဒ်** သည် အေဂျင့် တစ်ဦးချင်းစီ၏ တုံ့ပြန်မှုများကို သီးခြား စုဆောင်းသည်


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [attractions_agent, dining_agent, history_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Concurrent Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Architecture:</strong><br>
        • Input → <strong>Dispatcher</strong> (fan-out)<br>
        • <strong>3 Agents</strong> run in parallel (attractions, dining, history)<br>
        • Output → 3 AgentResponse objects, one per agent
    </p>
</div>
"""))

## Step 5: Test Case 1 - တိုကျို ခရီးသွားအကြံပြုချက်များ

တိုကျိုကို သွားသည့်နေရာအဖြစ် သုံးစုံလိုက်လုပ်ငန်းစဉ်ကို စမ်းသပ်ကြည့်အောင်။ အေးဂင့်(agents) သုံးယောက်လုံးက တပြိုင်နက် မည်သည့်ခရီးသွား အကြံပြုချက်များကို ပြည့်စုံစွာ ပေးရန် ပူးပေါင်း လုပ်ဆောင်မည်။


In [ ]:
async def display_travel_recommendations(destination: str):
    """Run the concurrent workflow and display formatted results."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Processing Travel Recommendations for {destination}</h3>
        <p style='margin: 0;'><strong>Status:</strong> Running 3 agents concurrently...</p>
    </div>
    """))

    # Run the workflow. With WorkflowBuilder(output_executors=[a1, a2, a3]),
    # outputs is a list of AgentResponse objects in the same order as output_executors.
    events = await workflow.run(f"I want comprehensive travel recommendations for {destination}")
    outputs = events.get_outputs()

    # Display results header
    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Complete Travel Guide for {destination}</h2>
        <p style='margin: 0; font-size: 14px; opacity: 0.9;'>Generated by 3 concurrent agents</p>
    </div>
    """))

    sections = [
        ("attractions-agent", AttractionsRecommendation, display_attractions_section),
        ("dining-agent", DiningRecommendation, display_dining_section),
        ("history-agent", HistoryRecommendation, display_history_section),
    ]

    for i, (agent_name, schema, render) in enumerate(sections):
        if i >= len(outputs):
            continue
        text = outputs[i].text
        try:
            data = schema.model_validate_json(text)
            render(data)
        except Exception as e:
            display(HTML(f"""
            <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>Error parsing {agent_name} response:</strong> {str(e)}
                <details><summary>Raw response</summary>{text}</details>
            </div>
            """))


def display_attractions_section(data: AttractionsRecommendation):
    """Display attractions recommendations in a formatted section."""
    attractions_list = "".join([f"<li>{attraction}</li>" for attraction in data.top_attractions])
    activities_list = "".join([f"<li>{activity}</li>" for activity in data.activities])

    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏛️ Tourist Attractions & Activities</h3>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Top Attractions:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{attractions_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
        <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Activities:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{activities_list}</ul>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Best Time to Visit:</strong> {data.best_time_to_visit}
        </div>
        <div>
            <strong style='color: #333;'>Transportation Tips:</strong> {data.transportation_tips}
        </div>
    </div>
    """))


def display_dining_section(data: DiningRecommendation):
    """Display dining recommendations in a formatted section."""
    dishes_list = "".join(
        [f"<li>{dish}</li>" for dish in data.must_try_dishes])
    restaurants_list = "".join(
        [f"<li>{restaurant}</li>" for restaurant in data.recommended_restaurants])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.food_experiences])

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🍜 Food & Dining Experiences</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Local Cuisine:</strong> {data.local_cuisine}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Must-Try Dishes:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{dishes_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Restaurants:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{restaurants_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Food Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <strong style='color: #333;'>Dining Etiquette:</strong> {data.dining_etiquette}
        </div>
    </div>
    """))


def display_history_section(data: HistoryRecommendation):
    """Display history recommendations in a formatted section."""
    highlights_list = "".join(
        [f"<li>{highlight}</li>" for highlight in data.cultural_highlights])
    periods_list = "".join(
        [f"<li>{period}</li>" for period in data.important_periods])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.cultural_experiences])
    facts_list = "".join(
        [f"<li>{fact}</li>" for fact in data.interesting_facts])

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>📚 History & Culture</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Historical Significance:</strong> {data.historical_significance}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Highlights:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{highlights_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Important Historical Periods:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{periods_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Interesting Facts:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{facts_list}</ul>
        </div>
    </div>
    """))


# Test with Tokyo
await display_travel_recommendations("Tokyo")

# အဆင့် ၆: စမ်းသပ်မှု ၂ - ပရင့်စ် ခရီးသွား အကြံပြုချက်များ


In [ ]:
await display_travel_recommendations("Paris")

## အဆင့် ၇: အစွမ်းထက်မှု विश्लेषण - တပြိုင်နက် vs ဆက်ပြီး

တပြိုင်နက် အလုပ်ဆောင်ခြင်းနှင့် ဆက်ပြီး အလုပ်ဆောင်ခြင်းတို့၏ အစွမ်းထက်မှု ကွာခြားချက်ကို တိုင်းတာပြီး တပြိုင်နက် အုပ်ချုပ်မှု၏ အကျိုးကျေးဇူးများကို ပြသကြစို့။


In [ ]:
import time


async def measure_concurrent_performance(destination: str):
    """Measure concurrent execution time."""
    start_time = time.time()

    events = await workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs)


async def measure_sequential_performance(destination: str):
    """Measure sequential execution time."""
    # Build a sequential workflow that chains the same agents one after another.
    sequential_workflow = (
        WorkflowBuilder(
            start_executor=attractions_agent,
            output_executors=[attractions_agent, dining_agent, history_agent],
        )
        .add_chain([attractions_agent, dining_agent, history_agent])
        .build()
    )
    start_time = time.time()

    events = await sequential_workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs)


async def performance_comparison():
    """Compare concurrent vs sequential performance."""
    test_destination = "Barcelona"

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Performance Comparison Test</h3>
        <p style='margin: 0;'>Testing with destination: <strong>Barcelona</strong></p>
    </div>
    """))

    # Test concurrent execution
    print("Running concurrent workflow...")
    concurrent_time, concurrent_count = await measure_concurrent_performance(test_destination)

    # Test sequential execution
    print("Running sequential workflow...")
    sequential_time, sequential_count = await measure_sequential_performance(test_destination)

    # Calculate performance improvement
    improvement = ((sequential_time - concurrent_time) / sequential_time) * 100

    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(102,126,234,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Performance Results</h2>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>⚡ Concurrent Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{concurrent_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{concurrent_count} agent responses</p>
            </div>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>🔄 Sequential Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{sequential_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{sequential_count} agent responses</p>
            </div>
        </div>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Performance Improvement</h4>
            <p style='margin: 0; font-size: 20px; font-weight: bold;'>{improvement:.1f}% faster</p>
            <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>
                Saved {sequential_time - concurrent_time:.2f} seconds with concurrent execution
            </p>
        </div>
    </div>
    """))


# Run performance comparison
await performance_comparison()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
